# Tasks 3.12–3.15 — Parallel HSMM Inference + Evaluation (Colab)

Notebook này chạy trên **Google Colab T4 GPU** và thực hiện Tasks 3.12–3.15:
- **Task 3.12:** Tính confidence C(ω) cho mỗi HSMM topology
- **Task 3.13:** Phân loại tiếng thổi mỗi bản ghi dựa trên C(M-N)
- **Task 3.14:** Tổng hợp dự đoán mỗi bệnh nhân
- **Task 3.15:** Đánh giá toàn bộ pipeline (weighted accuracy, confusion matrix, AUC-ROC)

## Cấu trúc Drive cần có
```
MyDrive/heart-murmur/
├── data/processed/spectrograms/   (3163 .npy files)
├── data/metadata/                 (patients.csv, recordings.csv, cv_splits.json)
├── models/rnn/                    (fold_0_best.pt ... fold_4_best.pt)
└── src/
    ├── __init__.py
    ├── data/__init__.py
    ├── models/
    │   ├── __init__.py
    │   ├── rnn.py
    │   ├── hsmm.py
    │   ├── viterbi.py
    │   └── parallel_hsmm.py
    └── features/
        ├── __init__.py
        └── spectrogram.py
```

## Trước khi chạy
1. **Runtime → Change runtime type → T4 GPU**
2. Đảm bảo Drive đã sync đủ các file trên
3. Chạy từng cell theo thứ tự từ trên xuống

## Setup — Kiểm tra GPU, Mount Drive, Copy data vào local SSD

In [ ]:
# [Setup Cell 1] Kiểm tra GPU và mount Google Drive
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: Khong co GPU — vao Runtime -> Change runtime type -> T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/heart-murmur')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Kiem tra cac file can thiet
print(f'\nKiem tra files:')
required = [
    'src/__init__.py',
    'src/data/__init__.py',
    'src/models/__init__.py',
    'src/models/rnn.py',
    'src/models/hsmm.py',
    'src/models/viterbi.py',
    'src/models/parallel_hsmm.py',
]
all_ok = True
for f in required:
    exists = (PROJECT_ROOT / f).exists()
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {f}')
    if not exists:
        all_ok = False

n_spec = len(list((PROJECT_ROOT / 'data/processed/spectrograms').glob('*.npy')))
print(f'  [{"OK" if n_spec == 3163 else "WARN"}] spectrograms: {n_spec}/3163 files')
n_models = len(list((PROJECT_ROOT / 'models/rnn').glob('*_best.pt')))
print(f'  [{"OK" if n_models == 5 else "MISSING"}] model checkpoints: {n_models}/5 files')

if not all_ok:
    print('\nWARNING: Co file bi thieu. Copy day du src/ len Drive truoc khi tiep tuc.')

In [ ]:
# [Setup Cell 2] Tao __init__.py neu chua co (can cho Python import)
import os

init_paths = [
    PROJECT_ROOT / 'src' / '__init__.py',
    PROJECT_ROOT / 'src' / 'data' / '__init__.py',
    PROJECT_ROOT / 'src' / 'models' / '__init__.py',
    PROJECT_ROOT / 'src' / 'features' / '__init__.py',
]
for p in init_paths:
    if not p.exists():
        p.touch()
        print(f'Created: {p.relative_to(PROJECT_ROOT)}')
    else:
        print(f'Exists : {p.relative_to(PROJECT_ROOT)}')

In [ ]:
# [Setup Cell 3] Copy spectrograms tu Drive vao /content/ (SSD local)
# Ly do: Doc file tu Drive qua mang rat cham (~1s/file x 3163 = >50 phut)
# Doc tu SSD local chi mat ~0.01s/file
import shutil

LOCAL_SPEC_DIR = Path('/content/spectrograms')

if not LOCAL_SPEC_DIR.exists():
    print('Copying spectrograms tu Drive -> /content/ ...')
    shutil.copytree(PROJECT_ROOT / 'data/processed/spectrograms', LOCAL_SPEC_DIR)
    n = len(list(LOCAL_SPEC_DIR.glob('*.npy')))
    print(f'Done: {n} files')
else:
    n = len(list(LOCAL_SPEC_DIR.glob('*.npy')))
    print(f'Da co san trong /content/: {n} files')

In [ ]:
# [Setup Cell 4] Import libraries va dinh nghia paths
import numpy as np
import pandas as pd
import json
import time
from tqdm import tqdm

MODELS_DIR  = PROJECT_ROOT / 'models' / 'rnn'
META_DIR    = PROJECT_ROOT / 'data' / 'metadata'
SPEC_DIR    = LOCAL_SPEC_DIR   # SSD local, khong phai Drive
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Load metadata
patients_df   = pd.read_csv(META_DIR / 'patients.csv')
recordings_df = pd.read_csv(META_DIR / 'recordings.csv')
if 'recording_id' not in recordings_df.columns:
    recordings_df['recording_id'] = recordings_df['wav_path'].apply(
        lambda p: Path(p).stem)

with open(META_DIR / 'cv_splits.json') as f:
    cv_splits = json.load(f)

print(f'patients_df  : {patients_df.shape}')
print(f'recordings_df: {recordings_df.shape}')
print(f'CV splits    : {len(cv_splits)} folds')

# Build rec_id -> fold mapping (moi ban ghi thuoc dung 1 val fold)
rec_to_fold = {}
for fold_name, fold_data in cv_splits.items():
    for rec_id in fold_data['val_recordings']:
        rec_to_fold[rec_id] = fold_name
print(f'Total val recordings: {len(rec_to_fold)}')

In [ ]:
# [Setup Cell 5] Load 5 RNN model checkpoints
from src.models.rnn import build_model

models = {}
for fold_idx in range(5):
    fold_name = f'fold_{fold_idx}'
    ckpt = torch.load(MODELS_DIR / f'{fold_name}_best.pt', map_location=DEVICE)
    m = build_model(seed=42).to(DEVICE)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    models[fold_name] = m
    print(f'Loaded {fold_name}: epoch={ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f}')

print(f'\nTat ca {len(models)} models san sang tren {DEVICE}')

## Task 3.12 + 3.13 — Inference: RNN Posteriors + Parallel HSMM + Classification

### Dang lam gi
Chay toan bo pipeline inference tren 3163 ban ghi:
1. RNN → posteriors (T, 5)
2. get_hsmm_params() → HR, systolic_interval, duration distributions
3. run_parallel_hsmm() → 4 topology x Viterbi → 4 confidence scores C(ω)
4. C(M-N) = C(best_murmur) - C(healthy) → quyet dinh moi ban ghi

### Task 3.12 — Confidence C(ω)
```
C(ω) = (1/T) * SUM P(q_t = q_hat_t^(ω) | x, θ)
```
Trace Viterbi path qua posteriors da duoc modify cho tung topology.
Ket qua so hoc tuong duong voi trace qua original posteriors.

### Task 3.13 — Classification moi ban ghi
```
C(M-N) = C(best_murmur) - C(healthy)
C(M-N) > 0  →  Phat hien tieng tho
C(M-N) <= 0 →  Binh thuong
```

### Uoc tinh thoi gian
~3163 ban ghi x ~0.3s/ban ghi = ~15-20 phut tren T4 GPU

In [ ]:
# [Task 3.12 + 3.13] Inference tren toan bo val set (3163 ban ghi)
from src.models.parallel_hsmm import run_parallel_hsmm

all_results = []
errors      = []
all_val_ids = list(rec_to_fold.keys())
print(f'Total val recordings: {len(all_val_ids)}')
print('Bat dau inference...')

t0 = time.time()

for rec_id in tqdm(all_val_ids, desc='RNN + HSMM inference'):
    fold_name = rec_to_fold[rec_id]
    model     = models[fold_name]

    try:
        # Buoc 1: RNN inference → posteriors (T, 5)
        spec = np.load(SPEC_DIR / f'{rec_id}.npy')   # (41, T)
        T    = spec.shape[1]
        feat = torch.FloatTensor(spec.T).unsqueeze(0).to(DEVICE)  # (1, T, 41)

        with torch.no_grad():
            logits = model(feat, [T])
        probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()  # (T, 5)

        # Buoc 2: Parallel HSMM → 4 confidence scores + Viterbi paths
        # (Tasks 3.11 + 3.12 chay ben trong run_parallel_hsmm)
        result = run_parallel_hsmm(probs)

        # Buoc 3: Task 3.13 — Classification moi ban ghi
        c_mn  = result['murmur_conf'] - result['healthy_conf']  # C(M-N)
        c_hat = max(result['all_confs'].values())                # C(omega_hat)

        # Lay nhan thuc
        pid         = int(rec_id.split('_')[0])
        pat         = patients_df[patients_df['patient_id'] == pid].iloc[0]
        true_murmur = pat['murmur']

        all_results.append({
            'recording_id':  rec_id,
            'patient_id':    pid,
            'true_murmur':   true_murmur,
            'fold':          fold_name,
            'c_healthy':     float(result['healthy_conf']),
            'c_holo':        float(result['all_confs']['Holosystolic']),
            'c_early':       float(result['all_confs']['Early-systolic']),
            'c_mid':         float(result['all_confs']['Mid-systolic']),
            'c_murmur_best': float(result['murmur_conf']),
            'best_model':    result['murmur_model'],
            'c_mn':          float(c_mn),
            'c_hat':         float(c_hat),
            'hr_bpm':        float(result['heart_rate_bpm']),
            # Task 3.13: quyet dinh moi ban ghi
            'rec_pred':      'Murmur' if c_mn > 0 else 'NoMurmur',
        })

    except Exception as e:
        errors.append((rec_id, str(e)))

elapsed = time.time() - t0
n = max(len(all_results), 1)
print(f'\nDone: {len(all_results)} ban ghi trong {elapsed:.0f}s ({elapsed/n:.2f}s/ban ghi)')
print(f'Errors: {len(errors)}')
if errors:
    print('5 loi dau:')
    for rec_id, err in errors[:5]:
        print(f'  {rec_id}: {err}')

# Luu ket qua recording-level
df_rec = pd.DataFrame(all_results)
df_rec.to_csv(RESULTS_DIR / 'recording_results.csv', index=False)
print(f'\nSaved: recording_results.csv ({len(df_rec)} rows)')

# Preview
print('\nPhan bo C(M-N) theo nhan thuc (recording level):')
print(f'{"Label":>10}  {"N":>5}  {"mean C(M-N)":>12}  {"C(M-N)>0":>10}  {"C(hat)<0.65":>12}')
print('-' * 58)
for label in ['Present', 'Unknown', 'Absent']:
    sub   = df_rec[df_rec['true_murmur'] == label]
    if len(sub) == 0:
        continue
    n_pos = (sub['c_mn'] > 0).sum()
    n_low = (sub['c_hat'] < 0.65).sum()
    print(f'{label:>10}  {len(sub):>5}  '
          f'{sub["c_mn"].mean():>+12.4f}  '
          f'{n_pos:>4}/{len(sub):<5}  '
          f'{n_low:>4}/{len(sub)}')

## Task 3.14 — Tong hop du doan moi benh nhan

### Dang lam gi
Gom tat ca ban ghi cua cung benh nhan lai va ap dung quy tac tong hop:

```
Neu BAT KY ban ghi nao co C(M-N) > 0  →  "Present"
Neu BAT KY ban ghi nao co C(ω̂) < 0.65 →  "Unknown"
Nguoc lai                               →  "Absent"
```

### Thu tu uu tien
Present > Unknown > Absent
(Phat hien tieng tho uu tien tuyet doi — bo sot nguy hiem hon phat hien nham)

### Nguong 0.65
C(ω̂) < 0.65 nghia la tat ca 4 HSMM deu co confidence thap →
tin hieu qua nhieu, HSMM khong the phan doan tin cay → "Unknown" 

In [ ]:
# [Task 3.14] Tong hop du doan moi benh nhan
QUALITY_THRESHOLD = 0.65  # nguong chat luong tin hieu

labels = ['Present', 'Unknown', 'Absent']
patient_preds = []

for pid, group in df_rec.groupby('patient_id'):
    true_label   = group['true_murmur'].iloc[0]

    # Quy tac tong hop
    has_murmur   = (group['c_mn'] > 0).any()
    has_low_conf = (group['c_hat'] < QUALITY_THRESHOLD).any()

    if has_murmur:
        pred = 'Present'
    elif has_low_conf:
        pred = 'Unknown'
    else:
        pred = 'Absent'

    patient_preds.append({
        'patient_id':    int(pid),
        'true_murmur':   true_label,
        'pred_murmur':   pred,
        'correct':       bool(true_label == pred),
        'n_recordings':  int(len(group)),
        'n_murmur_recs': int((group['c_mn'] > 0).sum()),
        'c_mn_max':      float(group['c_mn'].max()),
        'c_mn_mean':     float(group['c_mn'].mean()),
        'c_hat_min':     float(group['c_hat'].min()),
    })

df_pat = pd.DataFrame(patient_preds)
df_pat.to_csv(RESULTS_DIR / 'patient_results.csv', index=False)
print(f'Saved: patient_results.csv ({len(df_pat)} benh nhan)')

# Confusion matrix
print('\nConfusion matrix (rows=Predicted, cols=True):')
cm = pd.crosstab(
    df_pat['pred_murmur'],
    df_pat['true_murmur'],
    rownames=['Predicted'],
    colnames=['True'],
)
cm = cm.reindex(index=labels, columns=labels, fill_value=0)
print(cm)

# So sanh voi PLOS Table 1
print('\nPLOS Table 1 (target):')
print('                True: Present  Unknown  Absent')
print('Pred: Murmur        166       19     117')
print('Pred: Unknown         1       21      39')
print('Pred: No murmur      12       28     539')

## Task 3.15 — Danh gia pipeline day du

### Dang lam gi
Tinh tat ca metrics de so sanh voi ket qua cong bo trong bai bao.

### Metrics can tinh
| Metric | Target (PLOS) |
|--------|:---:|
| Weighted accuracy (challenge metric) | 0.798 |
| Sensitivity — Present | 92.7% |
| Sensitivity — Unknown | 30.9% |
| Sensitivity — Absent | 77.6% |
| AUC-ROC (binary, loai Unknown) | 0.947 |

### Weighted accuracy
Challenge metric voi weights: Present=5, Unknown=3, Absent=1.
Bo sot Present (tieng tho that) bi phat boi 5x — uu tien sensitivity cao.

In [ ]:
# [Task 3.15 - Phan 1] Weighted accuracy va per-class metrics
from sklearn.metrics import classification_report

y_true = df_pat['true_murmur'].tolist()
y_pred = df_pat['pred_murmur'].tolist()

# 1. Weighted accuracy (PhysioNet Challenge metric)
weights = {'Present': 5, 'Unknown': 3, 'Absent': 1}
total_w, correct_w = 0, 0
for true, pred in zip(y_true, y_pred):
    w = weights[true]
    total_w   += w
    if true == pred:
        correct_w += w
weighted_acc = correct_w / total_w

# 2. Per-class sensitivity, PPV, F1
print('Per-class Sensitivity (Recall):')
sens_dict = {}
for label in labels:
    tp   = sum(t == label and p == label for t, p in zip(y_true, y_pred))
    fn   = sum(t == label and p != label for t, p in zip(y_true, y_pred))
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    sens_dict[label] = sens
    target = {'Present': 0.927, 'Unknown': 0.309, 'Absent': 0.776}[label]
    diff   = sens - target
    print(f'  {label:>10}: {sens:.3f}  (target={target:.3f}, diff={diff:+.3f})  [{tp}/{tp+fn}]')

print(f'\nWeighted accuracy : {weighted_acc:.4f}')
print(f'Target (PLOS)     : 0.798')
print(f'Difference        : {weighted_acc - 0.798:+.4f}')
print(f'Trong dung sai    : {"YES (+/-0.03)" if abs(weighted_acc - 0.798) <= 0.03 else "NO — ngoai dung sai"}')

print('\nFull classification report:')
print(classification_report(y_true, y_pred, target_names=labels, digits=3))

In [ ]:
# [Task 3.15 - Phan 2] AUC-ROC (binary: Present vs Absent, loai Unknown)
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Chi giu Present va Absent (loai Unknown)
df_binary = df_pat[df_pat['true_murmur'].isin(['Present', 'Absent'])].copy()
y_binary  = (df_binary['true_murmur'] == 'Present').astype(int)
scores    = df_binary['c_mn_max']   # C(M-N) max qua cac ban ghi cua benh nhan

auc = roc_auc_score(y_binary, scores)
print(f'AUC-ROC (binary, Unknown removed): {auc:.4f}')
print(f'Target (PLOS)                    : 0.947')
print(f'Difference                       : {auc - 0.947:+.4f}')
print(f'Trong dung sai                   : {"YES (+/-0.03)" if abs(auc - 0.947) <= 0.03 else "NO"}')

# Ve ROC curve
fpr, tpr, thresholds = roc_curve(y_binary, scores)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='steelblue', linewidth=2,
        label=f'Parallel HSMM (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random guess')
ax.set_xlabel('1 - Specificity (FPR)')
ax.set_ylabel('Sensitivity (TPR)')
ax.set_title('ROC Curve — Murmur Detection (Present vs Absent)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_curve.png')

In [ ]:
# [Task 3.15 - Phan 3] Luu tat ca metrics vao JSON
import json as json_lib

all_metrics = {
    'weighted_accuracy':    float(weighted_acc),
    'auc_roc_binary':       float(auc),
    'n_patients':           int(len(df_pat)),
    'sensitivity': {
        'Present': float(sens_dict['Present']),
        'Unknown': float(sens_dict['Unknown']),
        'Absent':  float(sens_dict['Absent']),
    },
    'targets_plos': {
        'weighted_accuracy': 0.798,
        'auc_roc_binary':    0.947,
        'sensitivity_present': 0.927,
        'sensitivity_unknown': 0.309,
        'sensitivity_absent':  0.776,
    },
    'differences': {
        'weighted_accuracy': float(weighted_acc - 0.798),
        'auc_roc_binary':    float(auc - 0.947),
        'sensitivity_present': float(sens_dict['Present'] - 0.927),
        'sensitivity_unknown': float(sens_dict['Unknown'] - 0.309),
        'sensitivity_absent':  float(sens_dict['Absent'] - 0.776),
    },
    'confusion_matrix': cm.to_dict(),
}

with open(RESULTS_DIR / 'reproduction_metrics.json', 'w') as f:
    json_lib.dump(all_metrics, f, indent=2)

print('Saved: reproduction_metrics.json')
print('\nTom tat so sanh voi PLOS:')
print(f'{"Metric":>25}  {"Ours":>8}  {"PLOS":>8}  {"Diff":>8}  {"OK?":>6}')
print('-' * 65)
rows = [
    ('Weighted accuracy',    weighted_acc,          0.798, 0.030),
    ('AUC-ROC (binary)',     auc,                   0.947, 0.030),
    ('Sensitivity Present',  sens_dict['Present'],  0.927, 0.050),
    ('Sensitivity Unknown',  sens_dict['Unknown'],  0.309, 0.050),
    ('Sensitivity Absent',   sens_dict['Absent'],   0.776, 0.050),
]
for name, ours, target, tol in rows:
    diff = ours - target
    ok   = 'YES' if abs(diff) <= tol else 'NO'
    print(f'{name:>25}  {ours:>8.3f}  {target:>8.3f}  {diff:>+8.3f}  {ok:>6}')

## Download ket qua ve may local

Sau khi chay xong, download zip chua tat ca ket qua:
- `recording_results.csv` — confidence scores tung ban ghi
- `patient_results.csv` — du doan tung benh nhan
- `reproduction_metrics.json` — tat ca metrics so sanh voi PLOS
- `roc_curve.png` — ROC curve

Giai nen vao `experiments/results/` trong project local.

In [ ]:
# [Download] Zip va download ket qua
import shutil
from google.colab import files

zip_path = '/content/hsmm_results'
shutil.make_archive(zip_path, 'zip',
                    PROJECT_ROOT / 'experiments', 'results')

files.download(zip_path + '.zip')
print(f'Download started: hsmm_results.zip')
print('Giai nen vao experiments/results/ trong project local.')